In [ ]:
import time
import os
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException, NoSuchElementException


In [ ]:
def collect_hotel_links(driver, search_url):
    driver.get(search_url)
    time.sleep(5) 
    
    print(f"--- 1. STARTING LINK COLLECTION ---")
    all_links = []
    page = 1
    
    # UNLIMITED: Loop until the 'Next' button is gone or disabled
    while True:
        print(f"Scraping Hotel List Page {page}...")
        
        # Slow scroll to trigger lazy loading
        for i in range(1, 15): 
            try:
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight * arguments[0] / 15);", i)
                time.sleep(0.5)
            except: pass
            
        # Using CSS Selector is safer than looking for specific classes that might change
        # We look for any 'a' tag that has '/hotel/' in the href
        hotel_cards = driver.find_elements(By.CSS_SELECTOR, "a[href*='/hotel/']")
        
        current_page_links = []
        for card in hotel_cards:
            try:
                url = card.get_attribute("href")
                # Filter to ensure it's a real hotel page
                if url and "/hotel/" in url and "search?" not in url and "#" not in url:
                    current_page_links.append(url)
            except StaleElementReferenceException:
                continue 
        
        # Remove duplicates
        current_page_links = list(set(current_page_links))
        all_links.extend(current_page_links)
        print(f"Found {len(current_page_links)} hotels on this page.")
        
        # Try to click Next Page
        try:
            next_btn = None
            try: next_btn = driver.find_element(By.ID, "paginationNext")
            except: pass
            
            if not next_btn:
                try: next_btn = driver.find_element(By.CSS_SELECTOR, "button[aria-label='Next page']")
                except: pass
            
            if next_btn:
                if "disabled" in next_btn.get_attribute("class") or next_btn.get_attribute("aria-disabled") == "true":
                    print("End of list reached.")
                    break
                driver.execute_script("arguments[0].click();", next_btn)
                time.sleep(5)
                page += 1
            else:
                print("No 'Next' button found. Stopping collection.")
                break
        except:
            print("Error handling Next button. Stopping collection.")
            break
    
    unique_links = list(set(all_links))
    print(f"Collection Complete! Found {len(unique_links)} unique hotels.\n")
    return unique_links


In [ ]:
def scrape_single_hotel(driver, hotel_url):
    print(f"Navigating to: {hotel_url}")
    driver.get(hotel_url)
    time.sleep(3)
    
    hotel_data = []
    hotel_name = "Unknown Hotel"
    
    # 1. GET HOTEL NAME
    try:
        try:
            hotel_name = driver.find_element(By.CSS_SELECTOR, "[data-selenium='hotel-header-name']").text
        except:
            hotel_name = driver.find_element(By.TAG_NAME, "h1").text
    except:
        print(f"Could not find hotel name for {hotel_url}")

    print(f"Processing: {hotel_name}")

    # 2. SCROLL TO REVIEWS
    try:
        review_section = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "reviewSection"))
        )
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", review_section)
    except:
        # Fallback if ID is missing
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.6);")
    
    time.sleep(4)

    review_page = 1
    max_review_pages = 20 # Default fallback
    
    # --- DETECT TOTAL PAGES DYNAMICALLY ---
    try:
        print("  - Checking total pages...")
        # Look for numbers inside the review pagination container
        pagination_container = driver.find_elements(By.CSS_SELECTOR, "div[data-selenium='reviews-pagination'] span")
        if not pagination_container:
            pagination_container = driver.find_elements(By.CSS_SELECTOR, "#reviewSection .Review-paginator span")
            
        found_numbers = []
        for el in pagination_container:
            txt = el.text.strip()
            if txt.isdigit():
                found_numbers.append(int(txt))
        
        if found_numbers:
            max_review_pages = max(found_numbers)
            print(f"  - Detected {max_review_pages} total pages of reviews.")
        else:
            print("  - Could not detect max pages, using default limit.")
    except Exception as e:
        print(f"  - Error checking pages: {e}")
    # --------------------------------------
    
    while review_page <= max_review_pages:
        try:
            # Re-find cards every loop
            # Using CSS Selector is better for classes with spaces
            cards = driver.find_elements(By.CSS_SELECTOR, ".Review-comment")
            
            if not cards:
                print("  - No reviews loaded yet. Waiting...")
                time.sleep(2)
                cards = driver.find_elements(By.CSS_SELECTOR, ".Review-comment")
                if not cards: break

            # Scroll to last card to trigger any lazy loading/pagination
            try:
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", cards[-1])
                time.sleep(1)
            except: pass

            print(f"  - Page {review_page}/{max_review_pages}: Extracting {len(cards)} reviews...")
            
            for card in cards:
                try:
                    score = "N/A"
                    author = "Unknown"
                    comment = ""
                    
                    # Using CSS Selectors is safer than Class Name
                    try: score = card.find_element(By.CSS_SELECTOR, ".Review-comment-leftScore").text
                    except: pass
                    
                    try: author = card.find_element(By.CSS_SELECTOR, ".Review-comment-reviewer").text
                    except: pass
                    
                    # IMPROVED COMMENT EXTRACTION
                    try: 
                        # Try specific text class first
                        comment = card.find_element(By.CSS_SELECTOR, ".Review-comment-bodytext").text
                    except: 
                        try:
                            # Fallback: Try the parent body container
                            comment = card.find_element(By.CSS_SELECTOR, ".Review-comment-body").text
                        except: pass

                    hotel_data.append({
                        "Hotel Name": hotel_name,
                        "Hotel Link": hotel_url,
                        "User Score": score,
                        "Author": author,
                        "Comment": comment
                    })
                except StaleElementReferenceException:
                    continue 
        except Exception as e:
            print(f"Error extracting reviews on page {review_page}: {e}")
            break

        # 3. CLICK NEXT REVIEW PAGE (NUMBER STRATEGY)
        try:
            # If we reached the max detected page, stop
            if review_page >= max_review_pages:
                print("  - Reached max detected page. Moving to next hotel.")
                break
                
            next_page_num = review_page + 1
            print(f"  - Looking for page {next_page_num} button...")
            
            next_page_btn = None
            
            # Strategy: Look for the specific number in the pagination container
            xpath_selectors = [
                f"//div[@data-selenium='reviews-pagination']//*[text()='{next_page_num}']",
                f"//div[contains(@class, 'Review-paginator')]//*[text()='{next_page_num}']",
                f"//div[@id='reviewSection']//*[text()='{next_page_num}']"
            ]
            
            for xpath in xpath_selectors:
                try:
                    elements = driver.find_elements(By.XPATH, xpath)
                    for el in elements:
                        if el.is_displayed():
                            next_page_btn = el
                            break
                    if next_page_btn: break
                except: continue
                
            if next_page_btn:
                # Scroll into view
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_page_btn)
                time.sleep(1)
                
                # Click it
                driver.execute_script("arguments[0].click();", next_page_btn)
                
                # EXTRA: Click the parent box just to be safe
                try:
                    driver.execute_script("arguments[0].parentNode.click();", next_page_btn)
                except: pass

                time.sleep(4) # Wait for reload
                review_page += 1
            else:
                print(f"  - Page {next_page_num} button not found. Moving on.")
                break

        except Exception as e:
            print(f"  - Could not click next page: {e}")
            break
            
    return hotel_data


In [ ]:
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-notifications") # Blocks browser popups

driver = webdriver.Chrome(options=options)



In [ ]:
csv_filename = "agoda_full_data.csv"
start_url = "[https://www.agoda.com/search?city=13170&ds=Dj6tAmjYvw7e2vc%2F](https://www.agoda.com/search?city=13170&ds=Dj6tAmjYvw7e2vc%2F)"

# --- 1. RESUME CHECK ---
# Check if we already have scraped data
existing_links = set()
master_data = []

if os.path.exists(csv_filename):
    try:
        print(f"Found existing CSV: {csv_filename}")
        df_existing = pd.read_csv(csv_filename)
        if 'Hotel Link' in df_existing.columns:
            existing_links = set(df_existing['Hotel Link'].tolist())
            # Load existing data into memory so we don't lose it when saving
            master_data = df_existing.to_dict('records')
        print(f"Resuming... {len(existing_links)} hotels already scraped.")
    except Exception as e:
        print(f"Error reading existing CSV: {e}. Starting fresh.")

# --- 2. COLLECT LINKS ---
hotel_links = collect_hotel_links(driver, start_url)

if not hotel_links:
    print("No hotels found! Exiting.")
else:
    # UNLIMITED: Scrape ALL collected links
    # Filter out hotels we have already scraped
    hotels_to_scrape = [link for link in hotel_links if link not in existing_links]

    print(f"\n--- 2. STARTING DATA EXTRACTION ---")
    print(f"Total Hotels Found: {len(hotel_links)}")
    print(f"Already Scraped: {len(existing_links)}")
    print(f"Remaining to Scrape: {len(hotels_to_scrape)}")

    for i, link in enumerate(hotels_to_scrape):
        print(f"\n--- Hotel {i+1}/{len(hotels_to_scrape)} ---")
        try:
            reviews = scrape_single_hotel(driver, link)
            if reviews:
                master_data.extend(reviews)

                # --- AUTO-SAVE AFTER EVERY HOTEL ---
                df = pd.DataFrame(master_data)
                df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
                print(f"  -> Saved progress to {csv_filename} ({len(master_data)} total reviews)")
                # -----------------------------------
            else:
                print("No reviews extracted for this hotel.")
        except Exception as e:
            print(f"Skipping hotel due to error: {e}")

    print(f"Done! Final save complete.")


In [ ]:
driver.quit()
